# NagarSetu V2.0 — Structural Health Monitoring (SHM) Model Training
### Multi-Class Defect Detection (YOLOv8s) & Crack Segmentation (YOLOv8n-seg)
**Runtime Recommended:** GPU (T4 in Google Colab)

This notebook trains the production neural models for NagarSetu Phase 1 & 2:
- **infra_defects.pt**: 5-class detection (crack, spalling, exposed_rebar, corrosion, efflorescence)
- **crack_seg.pt**: Pixel-level concrete crack boundary polygon segmentation


In [ ]:
# 1. Verify GPU availability
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")


In [ ]:
# 2. Install Ultralytics and dependencies
!pip install -q "ultralytics==8.3.40" pillow opencv-python-headless


In [ ]:
# 3. Clone or setup NagarSetu repository
import os
if not os.path.exists("NagarsetuV2.0"):
    !git clone https://github.com/yajatkataria08-a11y/NagarsetuV2.0.git
    %cd NagarsetuV2.0
else:
    %cd NagarsetuV2.0


In [ ]:
# 4. Ingest and prepare normalized datasets with 80/10/10 zero-leakage split
!python ai_engine/infra/prepare_data.py --parents 80 --crops 4


## Run 1: Multi-Class Defect Detection (yolov8s.pt, 50 epochs, imgsz=640)
Detects: crack, spalling, exposed_rebar, corrosion, efflorescence

In [ ]:
# Train Multi-Class Defect Detector
!yolo detect train \
    data=ai_engine/infra/infra.yaml \
    model=yolov8s.pt \
    epochs=50 \
    imgsz=640 \
    batch=16 \
    project=runs/detect \
    name=infra_defects_run \
    exist_ok=True


## Run 2: Crack Boundary Segmentation (yolov8n-seg.pt, 50 epochs, imgsz=640)
Produces exact pixel-level crack masks for width and skeleton measurement in Phase 2.

In [ ]:
# Train Crack Boundary Segmenter
!yolo segment train \
    data=ai_engine/infra/crack_seg.yaml \
    model=yolov8n-seg.pt \
    epochs=50 \
    imgsz=640 \
    batch=16 \
    project=runs/segment \
    name=crack_seg_run \
    exist_ok=True


In [ ]:
# 7. Export and package weights for NagarSetu production (< 100 MB each)
import shutil
os.makedirs("ai_engine/models", exist_ok=True)

# Detection
best_det = "runs/detect/infra_defects_run/weights/best.pt"
if os.path.exists(best_det):
    shutil.copy(best_det, "ai_engine/models/infra_defects.pt")
    print("✅ Exported ai_engine/models/infra_defects.pt")

# Segmentation
best_seg = "runs/segment/crack_seg_run/weights/best.pt"
if os.path.exists(best_seg):
    shutil.copy(best_seg, "ai_engine/models/crack_seg.pt")
    print("✅ Exported ai_engine/models/crack_seg.pt")

!ls -lh ai_engine/models/*.pt
